# End-to-end classroom project · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 26. End-to-end classroom project

Clean, audit, summarize, visualize, and export a small dataset with an explanation of every choice.

Before you start: Core modules and plotting; keep raw and clean data distinct.

Official reference: https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html

### M26C01. Clean the raw table and keep an audit trail

Bring the cleaning skills together. Remove the known exact duplicate, normalize labels, preserve why a score is unavailable, and flag impossible attendance as missing. This exercise makes no assumption that an absent score equals zero.

**Read:** Why is Score_Status created before numeric conversion?

Answer: The original text distinguishes an absent student from an already missing score. Conversion alone would erase that distinction.

**Predict:** How many rows and missing numeric scores remain?

Answer: 12 unique students remain; two scores are missing numerically, S004 and S008.

**Explain:** Why not replace attendance 105 with 100 automatically?

Answer: The correct attendance is unknown. Capping would invent a value; flagging allows later correction from a source.



In [ ]:
# Predict before running: How many rows and missing numeric scores remain?
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

print(df[["Student_ID", "Name", "Campus", "Score", "Score_Status", "Attendance"]])
print("Rows:", len(df))
print("Unavailable scores:", df["Score"].isna().sum())

**Your turn:** Use the cleaned table to create a review list containing students with a missing Score, Study_Hours, or Attendance. Store it in result with columns Student_ID, Score, Study_Hours, and Attendance, then print result.

For the check, keep these result names: `result`.

**Hint:** Use isna().any(axis=1) across the three columns.

**Watch for this:** Do not discard the raw labels before you have recorded their meaning.

In [ ]:
# Change this code to complete the task.
# Question: Use the cleaned table to create a review list containing students with a missing Score, Study_Hours, or Attendance. Store it in result with columns Student_ID, Score, Study_Hours, and Attendance, then print result.
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

review_columns = ["Score", "Study_Hours", "Attendance"]
result = df.loc[df[review_columns].isna().any(axis=1), ["Student_ID"] + review_columns]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S004','S006','S008','S011']
print("Code checks passed. Review the written explanation as well.")

Review S004, S006, S008, and S011. Each record has at least one unavailable or invalid value. A review flag does not tell you how the missing value should be replaced.

**Quick check:** What evidence supports changing 105 to missing?

1. Attendance is defined as a percentage from 0 to 100
2. Every high value is an outlier to remove
3. Pandas does not support numbers above 100

Answer: 1. The column's documented range makes 105 invalid. Pandas itself has no such numeric limit.

**Explain what you learned:** Clean with documented rules and preserve the reasons for missing values.

### M26C02. Report a mean with its denominator

A group average is easier to interpret when its sample count is visible. Count all students and count recorded scores separately. Pandas mean skips missing numeric scores by default; say that in your report.

**Read:** How are Students and Recorded_Scores different?

Answer: Students counts rows. Recorded_Scores counts non-missing scores.

**Predict:** Which campus has fewer recorded scores than students?

Answer: Dubai has six students but only four recorded scores in the cleaned raw table.

**Explain:** May you report the mean as an average of all six Dubai students?

Answer: No. It is based on four recorded scores. The missing observations may also affect the interpretation.



In [ ]:
# Predict before running: Which campus has fewer recorded scores than students?
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

result = (df.groupby("Campus", as_index=False)
          .agg(Students=("Student_ID", "size"),
               Recorded_Scores=("Score", "count"),
               Mean_Score=("Score", "mean")))
print(result.round(2))

**Your turn:** Produce the same report by Program. Add Missing_Scores = Students - Recorded_Scores, then sort from highest to lowest Mean_Score.

For the check, keep these result names: `result`.

**Hint:** Use groupby('Program'), named aggregation, and a new column.

**Watch for this:** A polished mean without the number of observed values can mislead.

In [ ]:
# Change this code to complete the task.
# Question: Produce the same report by Program. Add Missing_Scores = Students - Recorded_Scores, then sort from highest to lowest Mean_Score.
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

result = (df.groupby("Program", as_index=False)
          .agg(Students=("Student_ID", "size"),
               Recorded_Scores=("Score", "count"),
               Mean_Score=("Score", "mean")))
result["Missing_Scores"] = result["Students"] - result["Recorded_Scores"]
result = result.sort_values("Mean_Score", ascending=False)
print(result.round(2))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Program'].tolist() == ['AI','IT']
assert result['Missing_Scores'].tolist() == [0,2]
assert np.isclose(result.iloc[0]['Mean_Score'],515/6)
assert np.isclose(result.iloc[1]['Mean_Score'],77.25)
print("Code checks passed. Review the written explanation as well.")

AI has six recorded scores and a mean of 85.83. IT has four recorded scores, two missing scores, and a mean of 77.25. These fictional data do not establish a general difference between programs.

**Quick check:** Which denominator does mean use here?

1. Every row, treating missing values as zero
2. Only non-missing score values in each group
3. The number of campuses

Answer: 2. mean skips missing numeric values by default. Reporting count beside it makes the denominator clear.

**Explain what you learned:** Attach counts and missingness to summary statistics.

### M26C03. Join, audit, and select a follow-up list

Combine the cleaned records with assessment results, checking both match quality and row counts. The follow-up rule below uses an observed Score under 75 or a missing assessment. Unknown scores need separate review, not automatic failure.

**Read:** Why is fillna(False) used in the score condition?

Answer: An unknown score should not be classified as below 75. Missing scores are handled by a separate review process.

**Predict:** Which student is selected because an assessment is missing?

Answer: S012, Lina.

**Explain:** Does excluding an unknown score mean the student passed?

Answer: No. It only means this specific rule does not classify that student. Missing data still need review.



In [ ]:
# Predict before running: Which student is selected because an assessment is missing?
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

joined = df.merge(assessment(), on="Student_ID", how="left",
                  validate="one_to_one", indicator=True)
mask = joined["Score"].lt(75).fillna(False) | joined["_merge"].eq("left_only")
result = joined.loc[mask, ["Student_ID", "Score", "Quiz", "_merge"]]
print(result)

**Your turn:** Create a separate audit with an outer merge. Show only unmatched IDs and whether each ID belongs to the student table or the assessment table.

For the check, keep these result names: `result`.

**Hint:** Use how='outer', indicator=True, then filter _merge != 'both'.

**Watch for this:** A missing comparison may propagate pd.NA; decide how the rule treats unknowns.

In [ ]:
# Change this code to complete the task.
# Question: Create a separate audit with an outer merge. Show only unmatched IDs and whether each ID belongs to the student table or the assessment table.
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

audit = df.merge(assessment(), on="Student_ID", how="outer",
                 validate="one_to_one", indicator=True)
result = audit.loc[audit["_merge"].ne("both"), ["Student_ID", "_merge"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S012','S013']
print("Code checks passed. Review the written explanation as well.")

S012 is student-only; S013 is assessment-only. Investigate both before treating the assessment file as complete.

**Quick check:** A missing score should automatically become a failing score.

1. True for every dataset
2. True whenever mean() is used
3. False; a documented rule or source correction is needed

Answer: 3. A missing observation is not an observed zero or a confirmed failure.

**Explain what you learned:** Keep classification rules separate from data-quality review.

### M26C04. Create a table, a plot, and a reproducible export

Finish with a compact report and a plot that answers a stated question. Keep the clean table and the summary as separate outputs. On a new run, rebuild the report from the source and the written cleaning rules.

**Read:** Why does the summary CSV keep its index?

Answer: Campus is stored in the pivot table index and carries meaning. Removing it would lose the campus labels.

**Predict:** How many mean values appear in this 2-by-2 table?

Answer: Four: AI and IT within Dubai and RAK.

**Explain:** What must accompany this chart in a report?

Answer: The number of recorded scores per group, the missing-score policy, and a note that these are fictional teaching data. Bar differences do not prove causation.



In [ ]:
# Predict before running: How many mean values appear in this 2-by-2 table?
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

result = df.pivot_table(index="Campus", columns="Program", values="Score",
                        aggfunc="mean", observed=True)
counts = df.pivot_table(index="Campus", columns="Program", values="Score",
                        aggfunc="count", observed=True)
print("Recorded mean scores:")
print(result.round(2))
print("Recorded score counts:")
print(counts)
ax = result.plot.bar(color=["#216b62", "#b58b45"], rot=0)
ax.set_title("Recorded mean score by campus and program")
ax.set_ylabel("Mean score (0–100)")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()
df.to_csv(OUTPUT_DIR / "capstone_cleaned_students.csv", index=False)
result.round(2).to_csv(OUTPUT_DIR / "capstone_score_summary.csv")
counts.to_csv(OUTPUT_DIR / "capstone_recorded_counts.csv")
print("Saved cleaned data, summary, and recorded counts in outputs/.")

**Your turn:** Make a Program-level report with columns Program, Recorded_Scores, and Mean_Score. Export it without a DataFrame index to capstone_program_report.csv. Read the exported file back into read_back and verify its columns and row count.

For the check, keep these result names: `read_back`.

**Hint:** Use groupby, agg, to_csv(index=False), and read_csv.

**Watch for this:** CSV does not preserve every pandas dtype. Recheck dtypes when loading it later.

In [ ]:
# Change this code to complete the task.
# Question: Make a Program-level report with columns Program, Recorded_Scores, and Mean_Score. Export it without a DataFrame index to capstone_program_report.csv. Read the exported file back into read_back and verify its columns and row count.
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

result = (df.groupby("Program", as_index=False)
          .agg(Recorded_Scores=("Score", "count"), Mean_Score=("Score", "mean")))
result["Mean_Score"] = result["Mean_Score"].round(2)
path = OUTPUT_DIR / "capstone_program_report.csv"
result.to_csv(path, index=False)
read_back = pd.read_csv(path)
print(read_back)
print("Columns preserved:", read_back.columns.tolist() == result.columns.tolist())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert read_back.columns.tolist() == ['Program','Recorded_Scores','Mean_Score']
assert read_back['Recorded_Scores'].sum() == 10
assert read_back.shape == (2,3)
print("Code checks passed. Review the written explanation as well.")

The CSV contains Program, Recorded_Scores, and Mean_Score, with no extra unnamed index column. Reading the file back checks the delivered table, not just the in-memory object.

**Quick check:** When is index=False appropriate for this Program report?

1. When all meaningful labels already live in columns
2. Whenever the table contains numbers
3. Only for Excel files

Answer: 1. This report has Program as a normal column. Its default numeric index adds no useful information to the CSV.

**Explain what you learned:** Deliver a report that another person can reproduce and interpret.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates